# 词袋与向量

 - jieba
 - 使用collection框架的counter来统计
 - 实现编号

In [1]:
chinese_docs = [
    "自然语言处理是人工智能的一个重要分支",
    "机器学习包括监督学习和无监督学习",
    "深度学习是机器学习的一个子领域",
    "计算机视觉让机器能够看懂图像",
    "自然语言处理和计算机视觉都是AI的重要方向"
]

In [3]:
import pkg_resources
import jieba
from collections import Counter

In [4]:
counter = Counter()

for sentance in chinese_docs:
    counter.update(jieba.cut(sentance))
    # for word in jieba.cut(sentance):
    #     counter.update(word)

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\ADMINI~1\AppData\Local\Temp\jieba.cache
Loading model cost 0.665 seconds.
Prefix dict has been built successfully.


In [7]:
word2id = {word: idx for idx, word in enumerate(counter.keys())}
id2word = {idx: word for idx, word in enumerate(counter.keys())}

### 上面的就是词袋

In [6]:
word2id["人工智能"]

3

# 向量化 (将编号转为向量)

## 1. one-hot (单热编码)

In [11]:
import torch
from torch.nn.functional import one_hot

input_ids = torch.tensor([2, 4, 1, 5, 20])
output = one_hot(input_ids)
output

tensor([[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]])

## 2. 词嵌入

 - 编码就是位置，会随机生成向量，指代这个编码
 - 但最初随机生成的向量没有办法对应编码， 需要训练对应
 - 训练方法: Word2vec
     - Skip-Gram: 
     - Continues Bag Of Word:
     - 其他

In [12]:
from torch.nn import Embedding

emb = Embedding(1000, 64)

# 练习

In [6]:
chinese_docs = [
    "自然语言处理是人工智能的一个重要分支",
    "机器学习包括监督学习和无监督学习",
    "深度学习是机器学习的一个子领域",
    "计算机视觉让机器能够看懂图像",
    "自然语言处理和计算机视觉都是AI的重要方向"
]

import pkg_resources
import jieba
from collections import Counter

def word_bow(corpus):
    # 统计词
    counter = Counter()
    for sentance in corpus:
        counter.update(jieba.lcut(sentance))
    # 构建词袋: 词袋大小、word2id、id2word
    len_wb = len(counter.keys())
    word2id = {word: idx for idx, word in enumerate(counter.keys())}
    id2word = {idx: word for idx, word in enumerate(counter.keys())}
    
    return len_wb, word2id, id2word

def sentence2ids(word2id, sentence):
    # 分词
    words = jieba.lcut(sentence)
    # 返回索引list
    return [word2id[word] for word in words]

def ids2sentence(id2word, ids):
    return "".join([id2word[idx] for idx in ids])


In [8]:
len_s, word2id, id2word = word_bow(chinese_docs)
ids = sentence2ids(word2id, "计算机视觉让机器能够看懂图像")
sentence = ids2sentence(id2word, ids)
sentence, ids

('计算机视觉让机器能够看懂图像', [17, 18, 19, 8, 20, 21, 22, 23])

In [11]:
import torch
from torch.nn import Embedding

In [16]:
emb_space = Embedding(len_s, 10)
ids_trans = torch.tensor(ids)
print(emb_space)
emb_space(ids_trans)

Embedding(27, 10)


tensor([[-2.6907e-01, -1.5741e+00, -1.7753e-01,  3.6202e-01,  3.8003e-01,
          3.8567e-01,  4.5882e-01,  6.3746e-01,  4.7336e-01, -6.4629e-01],
        [ 2.0458e-01,  9.9319e-01, -8.6419e-03,  1.2325e-02,  4.5484e-01,
          1.0513e-02,  2.0558e+00, -7.8070e-01,  1.2735e-01, -4.9195e-01],
        [ 1.3808e+00,  8.9856e-01, -2.2858e-01,  5.6696e-01, -4.4506e-01,
          4.6391e-01,  1.7493e+00,  9.9520e-01, -2.1985e-01,  6.4559e-01],
        [ 5.2419e-01, -6.0828e-01, -3.8249e-01, -1.4697e-01, -3.0553e+00,
         -1.2034e+00,  3.0591e-02,  5.3626e-01,  1.0571e+00, -4.5630e-02],
        [ 1.7252e+00,  7.5910e-01, -2.0075e-01, -9.1415e-01,  4.8303e-01,
         -2.7098e-02, -2.8047e-03, -5.2693e-02, -1.2722e-01, -2.5121e-02],
        [-5.4654e-02,  1.1235e+00, -6.3601e-01,  5.3614e-01,  7.6233e-01,
         -1.0268e+00,  3.3190e-01,  7.1366e-01,  4.3909e-01,  6.8223e-02],
        [-2.3486e-01, -1.9597e+00, -3.6441e-01, -2.8306e-01,  6.6911e-01,
         -5.4087e-01,  1.2998e+0